# BDC Satria Data 2026 — Tahap 2: Meta-Learner (Stacking)

**Input:** `oof_predictions_wide.csv` dari Tahap 1 (26.527 baris, 9 kolom probabilitas = 3 model x 3 kelas + true_label).

**Yang dilakukan notebook ini:**
1. Load & validasi OOF (jumlah baris lengkap, tidak ada NaN)
2. Baseline: F1 tiap model individual (dari OOF) + soft-voting (rata-rata probabilitas)
3. Latih & bandingkan 3 meta-learner: **Logistic Regression**, **Random Forest**, **XGBoost** — semua dievaluasi dengan StratifiedKFold 5-fold di level meta (supaya jujur, tidak overfit ke OOF)
4. Pilih yang terbaik, fit ulang di seluruh OOF, simpan ke disk (`joblib`) untuk dipakai di Tahap 4 (prediksi test)

**Catatan penting:** evaluasi meta-learner HARUS pakai CV, bukan fit+eval di data yang sama — kalau tidak, skornya akan terlihat lebih tinggi dari kenyataan (overfit).

Ringan — hanya CPU (sklearn/xgboost), selesai dalam hitungan menit.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[INFO] xgboost belum terinstall -- jalankan: pip install xgboost")
    print("       Notebook tetap jalan dengan LogReg + RandomForest saja.")

DATA_ROOT = Path(r"C:\Users\MyPC PRO\Downloads\BDC2026")
OUTPUT_DIR = DATA_ROOT / "outputs"

CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
SEED = 42

## 2. Load & Validasi OOF

In [ ]:
oof = pd.read_csv(OUTPUT_DIR / "oof_predictions_wide.csv")
print(f"Jumlah baris : {len(oof)} (harusnya 26527)")
print(f"Kolom        : {list(oof.columns)}")

assert oof.isna().sum().sum() == 0, "Ada NaN di OOF! Cek ulang Tahap 1."

PROB_COLS = [c for c in oof.columns if "_prob_" in c]
print(f"\nKolom fitur probabilitas ({len(PROB_COLS)}):")
for c in PROB_COLS:
    print("  ", c)
assert len(PROB_COLS) == 9, f"Harusnya 9 kolom probabilitas, ketemu {len(PROB_COLS)}"

X = oof[PROB_COLS].values
y = oof["true_label"].map({n: i for i, n in enumerate(CLASS_NAMES)}).values
print(f"\nX shape: {X.shape} | y shape: {y.shape}")
print(f"Distribusi label: {dict(zip(CLASS_NAMES, np.bincount(y)))}")

## 3. Baseline — Model Individual & Soft-Voting

In [ ]:
# Deteksi otomatis nama model dari kolom (prefix sebelum _prob_)
model_keys = sorted(set(c.split("_prob_")[0] for c in PROB_COLS))
print("Model terdeteksi:", model_keys)

print("\n=== Baseline: F1 tiap model individual (argmax dari probabilitas OOF-nya) ===")
individual_f1 = {}
for mk in model_keys:
    cols = [f"{mk}_prob_recyclable", f"{mk}_prob_electronic", f"{mk}_prob_organic"]
    preds = oof[cols].values.argmax(axis=1)
    f1 = f1_score(y, preds, average="macro")
    individual_f1[mk] = f1
    print(f"  {mk:20s}: {f1:.4f}")

print("\n=== Baseline: Soft-Voting (rata-rata probabilitas 3 model) ===")
avg_probs = np.zeros((len(oof), 3))
for mk in model_keys:
    cols = [f"{mk}_prob_recyclable", f"{mk}_prob_electronic", f"{mk}_prob_organic"]
    avg_probs += oof[cols].values
avg_probs /= len(model_keys)
voting_preds = avg_probs.argmax(axis=1)
voting_f1 = f1_score(y, voting_preds, average="macro")
print(f"  soft_voting          : {voting_f1:.4f}")

## 4. Latih & Bandingkan Meta-Learner (CV 5-fold di level meta)

`cross_val_predict` menghasilkan prediksi out-of-fold di level meta — tiap baris diprediksi oleh meta-learner yang tidak melihat baris itu saat fit. Ini evaluasi yang jujur.

In [ ]:
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

candidates = {
    "logreg": LogisticRegression(max_iter=2000, C=1.0, random_state=SEED),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=SEED, n_jobs=-1),
}
if HAS_XGB:
    candidates["xgboost"] = XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        objective="multi:softprob", num_class=3,
        random_state=SEED, n_jobs=-1, verbosity=0,
    )

meta_results = {}
for name, clf in candidates.items():
    preds = cross_val_predict(clf, X, y, cv=skf_meta, n_jobs=-1)
    f1 = f1_score(y, preds, average="macro")
    meta_results[name] = f1
    print(f"  {name:15s}: macro F1 = {f1:.4f}")

## 5. Rekap Semua — Siapa Terbaik?

In [ ]:
print("="*54)
print("REKAP LENGKAP (macro F1, semua dievaluasi dari OOF):")
print("="*54)
rows = []
for mk, f1 in individual_f1.items():
    rows.append((f"[individu] {mk}", f1))
rows.append(("[ensemble] soft_voting", voting_f1))
for name, f1 in meta_results.items():
    rows.append((f"[meta]     {name}", f1))

for name, f1 in sorted(rows, key=lambda r: r[1], reverse=True):
    print(f"  {name:32s}: {f1:.4f}")

best_meta_name = max(meta_results, key=meta_results.get)
print(f"\nMeta-learner terbaik: {best_meta_name} ({meta_results[best_meta_name]:.4f})")
print(f"vs model individual terbaik: {max(individual_f1.values()):.4f}")
print(f"vs soft voting             : {voting_f1:.4f}")

## 6. Fit Final Meta-Learner di Seluruh OOF + Simpan

Setelah tahu siapa terbaik dari CV, fit ulang di **seluruh** OOF (bukan per-fold) dan simpan — ini yang dipakai di Tahap 4 untuk prediksi test.

In [ ]:
best_meta = candidates[best_meta_name]
best_meta.fit(X, y)

meta_path = OUTPUT_DIR / "meta_learner.joblib"
joblib.dump({
    "model": best_meta,
    "name": best_meta_name,
    "prob_cols": PROB_COLS,        # urutan kolom fitur -- WAJIB sama persis saat prediksi test
    "class_names": CLASS_NAMES,
    "cv_macro_f1": meta_results[best_meta_name],
}, meta_path)
print(f"[SAVED] {meta_path}")
print(f"  Meta-learner : {best_meta_name}")
print(f"  CV macro F1  : {meta_results[best_meta_name]:.4f}")
print(f"  Urutan fitur : {PROB_COLS}")

# Classification report final (dari prediksi CV terjujur)
final_preds = cross_val_predict(best_meta, X, y, cv=skf_meta, n_jobs=-1)
print("\nClassification report (CV, meta-learner terbaik):")
print(classification_report(y, final_preds, target_names=CLASS_NAMES, digits=4))

## Langkah Selanjutnya
- **Tahap 3**: retrain 3 model final (SigLIP2, Swin-Base V2, ConvNeXt V2-Base) pakai 100% data train
- **Tahap 4**: prediksi 1.458 gambar test dgn 3 model final (preprocessing: `Resize(sisi pendek) + CenterCrop` — HARUS sama dgn validasi Tahap 1), probabilitasnya disusun dgn urutan kolom `prob_cols` di atas, lalu masukkan ke `meta_learner.joblib`
- **Tahap 5**: format `submission_NamaTim.csv` (kolom `id`, `predicted`; 0=Recyclable, 1=Electronic, 2=Organic)